# Sandbox Demo of PyFlow

The goal of this script is to generate a basic suite (definition file and .ecf job card) using the Pyflow wrapper and EcFlow.

# This code generates the .def file:

"""
ecflow_suite ❯❯❯ tree
.
├── def
│   └── suite.def
├── include
│   ├── envir-p1.h
│   ├── head.h
│   └── tail.h
└── scripts
    ├── family_A
    │   ├── family_Aa
    │   │   └── task_Aa1.ecf
    │   ├── family_Ab
    │   │   ├── task_Ab1.ecf
    │   │   └── task_Ab2.ecf
    │   └── task_A1.ecf
    └── family_B
        ├── family_Ba
        │   └── task_Ba1.ecf
        └── task_B1.ecf
"""

In [ ]:
# imports 
# set up paths and server

import datetime
import sys
import os
import pyflow as pf
scratchdir = os.path.join(os.path.abspath(''), 'scratch')
filesdir = os.path.join(scratchdir, 'files')
outdir = os.path.join(scratchdir, 'out')

if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)
    
server_host = 'localhost'
server_port = 22921 #Anna's personal Ursa EcFlow server port

In [ ]:
#Build the test suite

class WorkflowTask(pf.Task):
    def __init__(self, name: str, context: dict, **kwargs):
        super().__init__(name, variables=context['variables'], script=context['script'])

class WorkflowAnchorFamily(pf.AnchorFamily):
    def __init__(self, name: str, famAb_tasks: dict, **kwargs):
        #super().__init__(name, variables=context['variables'])
        task_triggers = {}
        for tt in famAb_tasks.keys(): #should a dict., keys are tasks, tt is the key 
            task_tc= famAb_tasks[tt] #get the context dict for this task
            task_triggers[tt] = WorkflowTask(tt, task_tc)
        self.task_triggers = task_triggers
    
class TestSuiteBuilder:
    def __init__(self, filesdir, outdir, number=100):
        with pf.Suite('testSuite', host=pf.LocalHost('localhost'),
                      files=os.path.join(filesdir, 'testSuite', 'scripts'),
                      home=outdir, NUMBER=number) as s: #remove NUMBER?

            # Create family_A as parent family
            with pf.AnchorFamily('A'):
                # Create family_Aa and its tasks
                with pf.AnchorFamily('Aa'):
                    tAa1c = {'variables': {'NUMBER': number + 1}, #c is context
                             'script': "echo family_Aa NUMBER=$NUMBER"}
                    tAa1 = WorkflowTask('Aa1', tAa1c) #this will be py ecflow. context is a dict
                    #need a workflow anchor family with name of family & context. 

                # Create family_Ab and its tasks (configuration)
                famAb_tasks= {
                        'Ab1': {'variables': {'NUMBER': number + 1},
                                'script': "echo family_Ab1 NUMBER=$NUMBER"},
                        'Ab2': {'variables': {'NUMBER': number /2 },
                                'script': "echo family_Ab2 NUMBER=$NUMBER"}
                    }
                #with WorkflowAnchorFamily('Ab', famAb_tasks):
                fam = WorkflowAnchorFamily('Ab', famAb_tasks) 
                #with pf.AnchorFamily('Ab'):
#
                    #task_triggers = {}
                    #for tt in famAb_tasks.keys(): #should a dict., keys are tasks, tt is the key 
                    #    task_tc= famAb_tasks[tt] #get the context dict for this task
                    #    task_triggers[tt] = WorkflowTask(tt, task_tc) #create the task
                        #i create famAb_task dict: with context and tasks/keys
                        #create list:
                        
                    
                # Create task A1 directly under family A
                tA1c = {'variables': {'NUMBER': number + 1}, #c is context
                         'script': "echo family_Aa NUMBER=$NUMBER"}
                tA1 = WorkflowTask('A1', tA1c) #this will be py ecflow. context is a dict

                # Optional - Set up task dependencies within family A:
                #look up other options for this:
                #tAa1 >> task_triggers['Ab1'] >>  task_triggers['Ab1']  >> tA1
                tAa1 >> fam.task_triggers['Ab1'] >>  fam.task_triggers['Ab1']  >> tA1

            # Create family_B as parent family
            with pf.AnchorFamily('B'):
                # Create family_Ba and its tasks
                with pf.AnchorFamily('Ba'):
                    tBa1c = {'variables': {'NUMBER': number / 4}, #c is context
                             'script': "echo family_Ba1 NUMBER=$NUMBER"}
                    tBa1 = WorkflowTask('Ba1', tBa1c) #this will be py ecflow. context is a dict

                tB1c = {'variables': {'NUMBER': number / 4}, #c is context
                         'script': "echo family_B1 NUMBER=$NUMBER"}
                tB1 = WorkflowTask('B1', tB1c) #this will be py ecflow. context is a dict
                
                #Optional - Set up task dependencies within family B:
                tB1 >> tBa1

        self.s = s
        self.s.check_definition()
        self.s.deploy_suite()
        def_dir = os.path.join(filesdir, 'testSuite', 'def')
        if not os.path.exists(def_dir):
            os.makedirs(def_dir, exist_ok=True)
        suite_def = self.s.ecflow_definition()
        suite_def.save_as_defs(os.path.join(def_dir, 'testSuite.def'))

# Instantiate using default 100
builder = TestSuiteBuilder(filesdir, outdir, 100)
print(builder.s)


suite testSuite
  edit NUMBER '100'
  edit ECF_FILES '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyflow/tutorials/course/scratch/scratch/files/testSuite/scripts'
  edit ECF_HOME '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyflow/tutorials/course/scratch/scratch/out'
  edit ECF_JOB_CMD 'bash -c 'export ECF_PORT=%ECF_PORT%; export ECF_HOST=%ECF_HOST%; export ECF_NAME=%ECF_NAME%; export ECF_PASS=%ECF_PASS%; export ECF_TRYNO=%ECF_TRYNO%; export PATH=/home/Anna.Smoot/.conda/envs/pf3.5.1/bin:$PATH; ecflow_client --init="$$" && %ECF_JOB% && ecflow_client --complete || ecflow_client --abort ' 1> %ECF_JOBOUT% 2>&1 &'
  edit ECF_KILL_CMD 'pkill -15 -P %ECF_RID%'
  edit ECF_STATUS_CMD 'true'
  edit ECF_CHECK_CMD 'true'
  edit ECF_OUT '%ECF_HOME%'
  label exec_host "localhost"
  family A
    edit ECF_FILES '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyflow/tutorials/course/scratch/scratch/files/testSuite/scripts/A'
    family Aa
      edit ECF_FILES '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyfl

In [ ]:

# scratch work for creating includes for single family and task 
# imports 
# set up paths and server

import datetime
import sys
import os
import pyflow as pf
scratchdir = os.path.join(os.path.abspath(''), 'scratch_include')
filesdir = os.path.join(scratchdir, 'files')
outdir = os.path.join(scratchdir, 'out')

if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)
    
server_host = 'localhost'
server_port = 22921 #Anna's personal EcFlow server port

In [ ]:
# Create basic suite

class WorkflowTask(pf.Task):
    def __init__(self, name: str, context: dict, **kwargs):
        super().__init__(name, variables=context['variables'], script=context['script'])
    
class TestSuiteBuilder:
    def __init__(self, filesdir, outdir, number=100):
        with pf.Suite('testSuite', host=pf.LocalHost('localhost'),
                      files=os.path.join(filesdir, 'testSuite', 'scripts'),
                      home=outdir, NUMBER=number) as s: #remove NUMBER?

            # Create family_Aa and its tasks
            with pf.AnchorFamily('A'):
                tAa1c = {'variables': {'NUMBER': number + 1}, #c is context
                            'script': "echo family_A NUMBER=$NUMBER"}
                tAa1 = WorkflowTask('A', tAa1c) #this will be py ecflow. context is a dict

        self.s = s
        self.s.check_definition()
        self.s.deploy_suite()
        
        # Create def directory and save suite definition there
        def_dir = os.path.join(filesdir, 'testSuite', 'def')
        if not os.path.exists(def_dir):
            os.makedirs(def_dir, exist_ok=True)
        suite_def = self.s.ecflow_definition()
        # Save definition to def directory
        suite_def.save_as_defs(os.path.join(def_dir, 'testSuite.def'))
        # Create include directory in the testSuite directory
        include_dir = os.path.join(filesdir, 'testSuite', 'include')
        if not os.path.exists(include_dir):
            os.makedirs(include_dir, exist_ok=True)

# Instantiate using default 100
builder = TestSuiteBuilder(filesdir, outdir, 100)
print(builder.s)

In [ ]:
#if user has .h files, copy them to include directory, otherwise use the canned .h files
import shutil
import os

# Define source and destination paths
source_dir = os.path.join(os.path.abspath(''), 'canned_include')
dest_dir = os.path.join(scratchdir, 'files', 'testSuite', 'include')

# List of .h files to copy
h_files = ['head.h', 'tail.h', 'envir-p1.h']

# Copy each .h file
for h_file in h_files:
    src_file = os.path.join(source_dir, h_file)
    dst_file = os.path.join(dest_dir, h_file)
    shutil.copy2(src_file, dst_file)
    print(f"Copied {h_file} to {dst_file}")